# 🖥️ SLIDE 1: GIỚI THIỆU ĐỀ TÀI & KHỞI TẠO DỰ ÁN (LAB 08)

### 📌 Đề tài: KTX Trường Bia — Sàn Ký Quỹ Mua Bán Đồ Cũ Sinh Viên (Chủ đề 1)
- **Bối cảnh:** Sinh viên KTX Trường Bia (Đại học Huế) có nhu cầu cao về mua bán, thanh lý đồ dùng học tập, nội trú.
- **Vấn đề cốt lõi:** Nguy cơ mất cọc, bị quỵt tiền hoặc bùng hàng khi giao dịch tự phát.
- **Giải pháp Smart Contract:** Ký quỹ độc lập on-chain trên mạng Sepolia, tiền chỉ giải ngân khi người mua đã kiểm tra và nhận đồ.
- **Quy ước dự án:** Áp dụng nghiêm ngặt `AGENTS.md` (Solidity ^0.8.20, CEI, custom error, không ghi API key).
- **Thành viên:** Lê Tiến Hùng (23K4300007) kiêm nhiệm 4 vai trò (Đặc tả, Hợp đồng, Kiểm thử, Giao diện).

# 🖥️ SLIDE 2: ĐẶC TẢ NGHIỆP VỤ & QUY TẮC KINH TẾ (LAB 08)

### 📋 5 Quy tắc nghiệp vụ (SPEC.md):
- **R1 (Nạp tiền):** Người mua nạp chính xác 100% giá niêm yết; chặn người bán tự mua đồ của mình.
- **R2 (Giải ngân):** Người mua xác nhận nhận đồ `confirmReceived()` -> Giải ngân 99% cho seller, 1% vào Quỹ KTX.
- **R3 (Chống giam vốn):** Quá 3 ngày kiểm tra, người bán có quyền gọi `claimPaymentAfterDeadline()` nhận tiền nếu người mua im lặng.
- **R4 (Khiếu nại):** Cho phép mở tranh chấp `raiseDispute()` trước hạn để đóng băng quỹ khi có hàng lỗi.
- **R5 (Trọng tài):** Trọng tài KTX phân xử dứt điểm (chuyển tiền cho bên thắng).

### 💡 Phản biện với AI:
- AI từng đề xuất hàm `refundAfterDeadline()` tự động hoàn tiền cho buyer. Sinh viên phát hiện lỗ hổng: người mua có thể cố tình im lặng để vừa lấy đồ vừa rút tiền cọc. Đã sửa thành tự động giải ngân cho seller.

# 🖥️ SLIDE 3: HỌC KỸ THUẬT KÉT KHÓA THỜI GIAN TIMELOCK (LAB 09)

### 🛡️ 4 Kỹ thuật Web3 cốt lõi từ `TimeLockVault.sol`:
1. **Checks - Effects - Interactions (CEI):** Kiểm tra điều kiện -> Emit sự kiện -> Chuyển ETH sau cùng chống Reentrancy.
2. **Custom Errors:** Dùng `error NotOwner()`, `error StillLocked(...)` thay cho require chuỗi dài để tối ưu gas.
3. **Chuyển ETH bằng .call:** `payable(owner).call{value: amount}("")` thay cho `transfer()` để tránh bẫy 2.300 gas.
4. **Sự kiện indexed:** Cho phép DApp và Etherscan lọc giao dịch theo địa chỉ ví nhanh chóng.

### 📊 Kết quả đo Gas trên Remix VM (Khóa 120s):
- **Deploy két:** ~200.000 gas (Thành công)
- **Nạp 1 ETH:** ~25.000 gas (Thành công, emit Deposited)
- **Rút khi còn khóa (t < 120s):** ~22.000 gas (Bị từ chối: StillLocked)
- **Rút sau 2 phút (t >= 120s):** ~30.000 gas (Thành công, nhận đủ 1 ETH)

# 🖥️ SLIDE 4: AUDIT MÃ NGUỒN AI & KHAI THÁC Ô NHỚ EVM (LAB 10)

### 🔬 Thực nghiệm đọc ô nhớ private bằng `eth_getStorageAt`:
- Hợp đồng `VaultBuggy.sol` lưu `uint256 private emergencyPin = 123456` tại Slot 0x2.
- Dùng RPC `eth_getStorageAt(contract, "0x2", "latest")` trên console -> đọc được hex `0x01e240` = 123456.
- **Bài học an ninh:** Từ khóa `private` chỉ giới hạn giữa các smart contract, mọi dữ liệu on-chain đều công khai 100%!

### 🛡️ Vá lỗi hợp đồng sản phẩm `ProjectCore.sol`:
- **Vá lỗi CEI:** Đưa lệnh `emit DisputeResolved(...)` lên trước lệnh gọi call chuyển tiền trong `_distributeFunds()`.
- **Vá lỗi xung đột lợi ích:** Chặn người bán kiêm trọng tài (`_seller != _arbiter`), chặn trọng tài tự mua hàng (`msg.sender != arbiter`).
- **Chuẩn hóa dung lượng:** Hợp đồng đạt đúng 146 dòng (<150 dòng theo quy định Sổ tay).

# 🖥️ SLIDE 5: SỨC KHỎE CODEBASE & LỘ TRÌNH KẾ TIẾP (LAB 11 – 15)

### ✅ Đánh giá mức độ sẵn sàng cho Gate Review 1 (Lab 12):
- Đầy đủ 4 tài liệu: `SPEC.md`, `ECONOMIC_RULES.md`, `PROJECT_PLAN.md`, `AI_JOURNAL.md`.
- `ProjectCore.sol` biên dịch sạch 0 lỗi với Solc 0.8.20+.
- Đồng bộ 100% nhánh main trên GitHub repo `lap8-15-baocao` kèm script `sync.bat` 1-click.

### 🎯 Lộ trình triển khai tiếp theo:
- **Lab 11:** Cài quy tắc kinh tế vào sản phẩm & bộ ca test vi phạm/thành công.
- **Lab 12:** Thuyết trình Gate Review 1 trước lớp (bảo vệ phạm vi dự án).
- **Lab 13:** Kiểm thử tấn công Reentrancy (vụ hack The DAO 2016) trên VulnerableBank và vá lỗi.
- **Lab 14 & 15:** Rà soát chéo giữa các nhóm, hoàn thiện giao diện DApp Web3 và triển khai GitHub Pages.